# Competencia: ¿qué está contratando el Estado? Segunda ronda

**Sesión 7 · IA Intermedio · EstudIA 2026**

Los mismos contratos públicos de la sesión 6 (SECOP II, 2025) y las mismas 15 categorías. **Los mismos 3.000 contratos
de prueba**, así que puedes comparar con la tabla de la sesión 6. Lo que cambia: ahora tienes **256 contratos
etiquetados por categoría** (3.840; incluyen los 480 de la sesión 6), y vale ajustar el transformer.

La pregunta: con más ejemplos, ¿el ajuste fino le gana a los embeddings congelados?

**La métrica: exactitud**, como en la sesión 6 (200 contratos de prueba por categoría).

**Las líneas de referencia.** En la sesión 6, el mejor modelo de referencia llegó a 78,7 % en la tabla con 32
ejemplos por categoría. Una **bolsa de palabras** entrenada con los 3.840 contratos llega a 79,5 %: esa es la
línea que hay que superar.

**Reglas**

1. Vale todo lo que hemos visto: ajuste fino (todo el modelo, solo la cabeza o LoRA), embeddings congelados con un
   clasificador encima, o una combinación. No valen los modelos que generan texto, como los chats (llegan en la
   sesión 9).
2. Tienes 3.840 contratos con su categoría y 3.000 sin categoría. Valida con una parte de tus 3.840.
3. Envía tus predicciones todas las veces que quieras (una vez cada 10 segundos). Gana la mayor **exactitud**.
4. La tabla en vivo usa **la mitad** de los contratos de prueba. Al final, el profesor revela el resultado con **la
   otra mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. Activa una GPU si puedes (*Entorno de ejecución › Cambiar
tipo de entorno de ejecución › GPU T4*): con GPU, el primer entrenamiento tarda unos minutos; sin GPU, mucho más. Ejecuta
todo el notebook (*Entorno de ejecución › Ejecutar todas*). Cuando te pida la clave, cópiala de la página de la
competencia en la plataforma y pégala.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os, re, time, json, urllib.request, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s07")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

try:
    from sentence_transformers import SentenceTransformer
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed, pipeline)
from transformers.utils import logging as registro_hf
registro_hf.set_verbosity_error()   # sin avisos largos al cargar los modelos
try:
    from peft import LoraConfig, get_peft_model
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
    from peft import LoraConfig, get_peft_model

def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

class Ajuste:
    """Lo que queda de un ajuste fino: el entrenador, el tokenizador, las clases y unos datos para el registro."""
    def __init__(self, **datos):
        self.__dict__.update(datos)

def _ejemplos(tok, textos, largo, clases=None, etiquetas=None):
    """Convierte los textos en tokens (cortando en `largo`) y, si hay etiquetas, les pone su número de clase."""
    cod = tok([str(t) for t in textos], truncation=True, max_length=largo)
    filas = [{k: cod[k][i] for k in cod} for i in range(len(textos))]
    if etiquetas is not None:
        for fila, e in zip(filas, etiquetas):
            fila["labels"] = clases.index(str(e))
    return filas

def ajustar(textos, etiquetas, modelo, forma="todo", tasa=5e-5, epocas=3, lote=16, largo=128,
            val_textos=None, val_etiquetas=None, semilla=0, mostrar=True):
    """Ajuste fino de un modelo de texto de Hugging Face para clasificar, con el Trainer.

    forma: "todo" (todo el modelo), "cabeza" (solo la capa nueva de encima) o "lora" (matrices pequeñas añadidas).
    Si le das textos de validación, mide la exactitud de validación al final de cada época."""
    set_seed(semilla)
    clases = sorted({str(e) for e in etiquetas})
    tok = AutoTokenizer.from_pretrained(modelo)
    red = AutoModelForSequenceClassification.from_pretrained(
        modelo, num_labels=len(clases), id2label=dict(enumerate(clases)), label2id={c: i for i, c in enumerate(clases)})
    if forma == "cabeza":
        for nombre, p in red.named_parameters():
            p.requires_grad = nombre.startswith("classifier")
    elif forma == "lora":
        red = get_peft_model(red, LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1))
    entrenables = sum(p.numel() for p in red.parameters() if p.requires_grad)
    total = sum(p.numel() for p in red.parameters())
    hay_val = val_textos is not None
    args = TrainingArguments(output_dir="entrenamiento", per_device_train_batch_size=lote,
                             per_device_eval_batch_size=64, learning_rate=tasa, num_train_epochs=epocas,
                             warmup_steps=0.1, weight_decay=0.01, eval_strategy="epoch" if hay_val else "no",
                             logging_strategy="epoch", save_strategy="no", report_to="none", seed=semilla,
                             fp16=torch.cuda.is_available(), disable_tqdm=not mostrar)
    entrenador = Trainer(
        model=red, args=args, data_collator=DataCollatorWithPadding(tok),
        train_dataset=_ejemplos(tok, textos, largo, clases, etiquetas),
        eval_dataset=_ejemplos(tok, val_textos, largo, clases, val_etiquetas) if hay_val else None,
        compute_metrics=lambda p: {"exactitud": float((p.predictions.argmax(-1) == p.label_ids).mean())})
    inicio = time.time()
    entrenador.train()
    return Ajuste(entrenador=entrenador, tok=tok, clases=clases, largo=largo, segundos=time.time() - inicio,
                  entrenables=entrenables, total=total, modelo=modelo, forma=forma)

def predecir(ajuste, textos):
    """La clase que el modelo ajustado le da a cada texto."""
    salida = ajuste.entrenador.predict(_ejemplos(ajuste.tok, textos, ajuste.largo))
    return np.array(ajuste.clases)[salida.predictions.argmax(-1)]

def dibujar_historia(ajuste, titulo="Cómo fue el entrenamiento"):
    """Las curvas de la sesión 2: pérdida de entrenamiento y de validación, y exactitud de validación, por época."""
    h = ajuste.entrenador.state.log_history
    ent = [(r["epoch"], r["loss"]) for r in h if "loss" in r]
    val = [(r["epoch"], r["eval_loss"], r["eval_exactitud"]) for r in h if "eval_loss" in r]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3))
    a.plot([e for e, _ in ent], [l for _, l in ent], "o-", color="#808080", label="Entrenamiento")
    if val:
        a.plot([v[0] for v in val], [v[1] for v in val], "o-", color="#CB318E", label="Validación")
        b.plot([v[0] for v in val], [v[2] for v in val], "o-", color="#CB318E")
        for e, _, x in val:
            b.annotate(pct(x), (e, x), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9)
        b.set_ylabel("Exactitud de validación"); b.set_ylim(min(v[2] for v in val) - 0.1, 1)
    a.set_ylabel("Pérdida"); a.legend(frameon=False)
    for eje in (a, b):
        eje.set_xlabel("Época"); eje.spines[["top", "right"]].set_visible(False); eje.grid(alpha=0.3)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def millones(n):
    """118_000_000 -> "118 millones"; 12_345 -> "12.345"."""
    return f"{n / 1e6:.1f} millones".replace(".0 ", " ").replace(".", ",") if n >= 1e6 else miles(n)

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

datos = pd.read_csv(descargar("contratos_entrenamiento.csv"))
prueba = pd.read_csv(descargar("contratos_prueba.csv"))
CLASES = sorted(datos.categoria.unique())
GPU = torch.cuda.is_available()
print(f"Listo: {miles(len(datos))} contratos con categoría, {miles(len(prueba))} de prueba, {len(CLASES)} categorías.",
      f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: el ajuste fino va a tardar bastante.")

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud en la tabla.

    ids: los ids de los contratos de prueba. predicciones: una categoría por contrato, como "salud".
    nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s07/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return None
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return None
    metrica = res.get("metrica", "Exactitud")
    print(f"Envío {res['envio']} recibido. {metrica} en la tabla: {pct(res.get('puntaje', res['exactitud']))}.")
    print(f"Tu mejor resultado: {pct(res['mejor'])} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")
    return res

---
## 1. Validar

Con 3.840 ejemplos ya no hace falta la validación cruzada de la sesión 6: apartamos el 20 % (768 contratos, 51 por
categoría) para validar y entrenamos con el resto. Con 768 contratos, una diferencia de un punto todavía puede ser
suerte; una de tres o más, probablemente no.

In [ ]:
ent, val = train_test_split(datos, test_size=0.2, stratify=datos.categoria, random_state=0)
for _, fila in datos.sample(5, random_state=3).iterrows():
    print(f"{fila.categoria:15s} {fila.texto[:100]}")
print(f"\nEntrenamiento: {miles(len(ent))} · validación: {miles(len(val))}")

bolsa = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2))
pred = LogisticRegression(max_iter=3000, C=10).fit(bolsa.fit_transform(ent.texto), ent.categoria).predict(
    bolsa.transform(val.texto))
LINEA_BASE = (pred == val.categoria.to_numpy()).mean()
print("Bolsa de palabras · exactitud de validación:", pct(LINEA_BASE))
registrar(modelo="bolsa de palabras", forma="", minusculas="no", tasa=None, epocas=None, ejemplos=len(ent),
          validacion=round(100 * LINEA_BASE, 1), segundos=None, tabla=None)

---
## 2. Tu modelo

Las piezas que puedes cambiar:

- `MODELO`: cualquier modelo de texto de Hugging Face que sirva para clasificar (tipo BERT).
- `FORMA`: `"todo"`, `"cabeza"` o `"lora"`, como en la demostración.
- `MINUSCULAS`: si es `True`, pasamos los textos a minúsculas antes de entrenar.
- `TASA`: la tasa de aprendizaje. `EPOCAS`: cuántas veces pasa por los ejemplos. `LARGO`: tokens por texto.
- `TODOS`: si es `True`, entrena con los 3.840 contratos (sin validación) para tu envío final.

Así como vienen funcionan, pero dejan mucho por ganar. Ejecútalas tal cual para tu primer envío y después mejóralas.

**Sin GPU**, el ajuste fino de los 3.072 contratos tarda casi media hora. En ese caso `FORMA` empieza en
`"congelados"`: los embeddings congelados de la sesión 6, ahora con 3.840 ejemplos, que funcionan sin GPU. También es
la comparación honesta: ¿cuánto gana el ajuste fino frente a esto?

In [ ]:
MODELO = "intfloat/multilingual-e5-small"
FORMA = "todo" if GPU else "congelados"     # "todo", "cabeza", "lora" o "congelados"
MINUSCULAS = False
TASA = 5e-5
EPOCAS = 3
LARGO = 128
TODOS = False

In [ ]:
def preparar(textos):
    return [t.lower() if MINUSCULAS else t for t in textos]

class Congelados:
    """Embeddings congelados + regresión logística (sesión 6), con la misma forma de usar que un ajuste."""
    def __init__(self, textos, etiquetas):
        self.e = SentenceTransformer(MODELO, device="cuda" if GPU else "cpu")
        self.prefijo = "query: " if "e5" in MODELO else ""
        inicio = time.time()
        self.clf = LogisticRegression(max_iter=5000, C=10).fit(self.vectores(textos), etiquetas)
        self.segundos = time.time() - inicio
    def vectores(self, textos):
        return self.e.encode([self.prefijo + t for t in textos], batch_size=64, normalize_embeddings=True,
                             show_progress_bar=False)
    def predecir(self, textos):
        return self.clf.predict(self.vectores(textos))

def entrenar(textos, etiquetas, **validacion):
    if FORMA == "congelados":
        return Congelados(preparar(textos), etiquetas)
    return ajustar(preparar(textos), etiquetas, MODELO, forma=FORMA, tasa=TASA, epocas=EPOCAS, largo=LARGO,
                   **validacion)

def predecir_con(modelo, textos):
    return modelo.predecir(preparar(textos)) if FORMA == "congelados" else predecir(modelo, preparar(textos))

if TODOS:
    modelo = entrenar(datos.texto, datos.categoria)
    EXACTITUD_VALIDACION = None
    print(f"Entrenado con los {miles(len(datos))} contratos en {modelo.segundos:.0f} s.")
else:
    validacion = {} if FORMA == "congelados" else {"val_textos": preparar(val.texto), "val_etiquetas": val.categoria}
    modelo = entrenar(ent.texto, ent.categoria, **validacion)
    EXACTITUD_VALIDACION = (predecir_con(modelo, val.texto) == val.categoria.to_numpy()).mean()
    print(f"Entrenado en {modelo.segundos:.0f} s. Exactitud de validación: {pct(EXACTITUD_VALIDACION)} · "
          f"bolsa de palabras: {pct(LINEA_BASE)}")
    if FORMA != "congelados":
        dibujar_historia(modelo)

### 2.1 Predecir la prueba y enviar

El envío queda en tu registro con lo que probaste y lo que sacó en la tabla. La primera vez te pide la clave.

In [ ]:
predicciones = predecir_con(modelo, prueba.texto)
nota = f"{MODELO.split('/')[-1]} {FORMA} min={MINUSCULAS}"
if FORMA != "congelados":
    nota += f" tasa={TASA} ep={EPOCAS}"
respuesta = enviar(prueba.id, predicciones, nota=nota + (" todos" if TODOS else ""))
registrar(modelo=MODELO, forma=FORMA, minusculas="sí" if MINUSCULAS else "no", tasa=None if FORMA == "congelados" else TASA,
          epocas=None if FORMA == "congelados" else EPOCAS, ejemplos=len(datos) if TODOS else len(ent),
          validacion=None if EXACTITUD_VALIDACION is None else round(100 * EXACTITUD_VALIDACION, 1),
          segundos=round(modelo.segundos), tabla=round(100 * respuesta["puntaje"], 1) if respuesta else None)
ver_registro()

---
## 3. Pistas para subir en la tabla

Prueba una cosa a la vez, mira la exactitud de **validación** y tu registro, y envía cuando mejore.

<details><summary>Pista 1</summary>

La misma de la sesión 6: ¿cómo están escritos los contratos? Casi todo lo que leyó el modelo estaba en minúsculas.
</details>

<details><summary>Pista 2</summary>

Mira las curvas: si la exactitud de validación sigue subiendo en la última época, prueba más épocas (5). Para un
modelo pequeño como e5-small, una tasa un poco más grande (`1e-4`) suele ir bien; para los modelos de 300 millones de
parámetros o más, más pequeña (`2e-5` o `3e-5`).
</details>

<details><summary>Pista 3</summary>

Cuando tengas una buena receta, entrénala con **todos** los contratos (`TODOS = True`): 768 ejemplos más. Ya no tendrás
validación, así que hazlo solo con una receta que ya validaste.
</details>

<details><summary>Pista 4</summary>

Otros modelos: `"dccuchile/bert-base-spanish-wwm-uncased"` (BETO, en español y ya en minúsculas, 110 millones de
parámetros), `"intfloat/multilingual-e5-base"` (278 millones) y `"FacebookAI/xlm-roberta-large"` (560 millones, solo con
GPU). Con los grandes, prueba `FORMA = "lora"` con una tasa más grande (`5e-4`).
</details>

<details><summary>Pista 5</summary>

La comparación honesta: `FORMA = "congelados"` usa los embeddings congelados de la sesión 6 con tus 3.840 contratos
(funciona sin GPU). Con `MINUSCULAS = True` y un modelo de embeddings más grande, como
`"intfloat/multilingual-e5-large"`, ¿hasta dónde llega? ¿Le gana tu mejor ajuste fino?
</details>

<details><summary>Pista 6</summary>

Junta dos modelos: promedia las probabilidades que da cada uno. Con `modelo.entrenador.predict(...)` obtienes los
números de cada clase antes de elegir la mayor.
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando el modelo a esa mitad
de la prueba. El resultado final se calcula con la otra mitad.

**Tu registro.** `experimentos.csv` desaparece cuando se reinicia Colab. Para guardarlo, descárgalo desde el panel de
archivos (a la izquierda).

**Fuente de los datos:** SECOP II – Contratos Electrónicos, Agencia Nacional de Contratación Pública – Colombia
Compra Eficiente, publicado en datos.gov.co. Licencia CC BY-SA 4.0. Usamos el objeto de contratos firmados en 2025;
la categoría es el segmento del código UNSPSC principal que eligió la entidad. Quitamos números de proceso, textos que
parecían nombrar a una persona, duplicados, las etiquetas evidentemente equivocadas y los contratos casi iguales a uno
de prueba.